# Bangladesh: 0209_change_detection_zscore

Static adaptation only; Earth Engine/data/asset execution not verified.

Original scientific assumptions and date windows retained; use the national workflow for QA improvements.

Review scale, missing data, units, and any training/calibration requirements before interpreting results.

May contain tutorial-specific assets or imported training features; inspect dependencies before execution.

In [ ]:
# Bangladesh adaptation: 0209_change_detection_zscore
# Original author: Amirhossein Ahrari; see original attribution below and originals/.
from bdgee.core import initialize, boundaries
from bdgee.notebook import export_outputs, upload_local
initialize()  # Run `earthengine authenticate` first; project is configured in bdgee.
BD_COUNTRY, BD_DISTRICTS = boundaries()
BD_ROI = BD_COUNTRY.geometry()
BD_BBOX = [88.0, 20.5, 92.7, 26.7]  # search only, not analysis boundary
roi = geometry = border = iran = tehran = province = loc = point = BD_ROI
# Run from the repository root; all local outputs are isolated by source.
from pathlib import Path
import os
_REPO_ROOT = Path.cwd()
_OUTPUT_DIR = _REPO_ROOT / 'outputs' / 'landcover' / '0209_change_detection_zscore'
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(_OUTPUT_DIR)


In [ ]:
"""00008_change_detection_zscore.ipynb

Tutorial Code by Amirhossein Ahrari
YouTube: https://www.youtube.com/@amirhosseinahrarigee
Tutorial Video: Google Earth Engine Tutotrial-197: Change Detection using Z-Score Analysis
This code is part of a tutorial series on Earth Engine programming techniques
presented on the Amirhossein Ahrari YouTube channel. You are free to use and modify
this code for academic and non-academic purposes. Don't forget to subscribe to
the Amirhossein Ahrari channel and follow the videos to support the instructor


"""
import ee
None
None
'# **Study Area**'
import geemap
map = geemap.Map(basemap='SATELLITE')
map
roi = BD_ROI
roi
'# **Sentinel-2 Images**'

def ndvi(img):
    bands = img.select('B.*').multiply(0.0001)
    index = bands.normalizedDifference(['B8', 'B4']).rename('ndvi')
    return index
sen2_2017 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterDate('2017', '2018').filterBounds(roi).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)).map(ndvi).median().rename('ndvi2017')
sen2_2025 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterDate('2025', '2026').filterBounds(roi).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)).map(ndvi).median().rename('ndvi2025')
stack = ee.Image.cat([sen2_2017, sen2_2025])
stack
'# **From Image to Xarray**'
from shapely.geometry import shape
from xee import helpers
sample = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterDate('2017', '2018').filterBounds(roi).first()
sample.select('B8').projection()
grid_params = helpers.fit_geometry(geometry=shape(roi.getInfo()), grid_crs='EPSG:32638', grid_scale=(10, -10))
import xarray as xr
ds = xr.open_dataset(stack, engine='ee', **grid_params)
ds = ds.squeeze('time').drop_vars('time') * 1
'# **Change Detection**'
ds['dif'] = ds['ndvi2025'] - ds['ndvi2017']
dif_mean = ds['dif'].mean(dim=['y', 'x']).item()
dif_std = ds['dif'].std(dim=['y', 'x']).item()
ds['z_score'] = (ds['dif'] - dif_mean) / dif_std
ds
'# **Visualization**'
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
p1 = ds['ndvi2017'].plot(ax=ax[0], x='x', y='y', robust=True, cmap='RdYlGn')
p2 = ds['ndvi2025'].plot(ax=ax[1], x='x', y='y', robust=True, cmap='RdYlGn')
p3 = ds['z_score'].plot(ax=ax[2], x='x', y='y', robust=True)
ax[0].set_title('2017', fontsize=20, fontweight='bold')
ax[1].set_title('2015', fontsize=20, fontweight='bold')
ax[2].set_title('Change', fontsize=20, fontweight='bold')
plt.tight_layout()
plt.savefig('ndvi_map.png', dpi=360, bbox_inches='tight')
'# **Change Area**'
inc = (ds['z_score'] <= -1).astype(int)
inc.plot(x='x', y='y')
import rioxarray
x_res, y_res = ds.rio.resolution()
y_res
pixel_area = abs(x_res * y_res)
pixel_area
(inc == 1).sum(dim=['x', 'y']).item() * pixel_area / 1000000.0


In [ ]:

# Review these task plans; task creation does not start exports.
os.chdir(_REPO_ROOT)
bd_tasks = export_outputs(globals(), 'landcover', '0209_change_detection_zscore', scale=1000)
# Set an appropriate native scale above before creating plans.
# for task in bd_tasks: task.start()
# upload_local(_OUTPUT_DIR, 'landcover')  # explicit upload; requires Drive OAuth scope
